In [ ]:
import pandas as pd
from pathlib import Path
import shutil

# --- CONFIGURATION ---
MASTER_CLASS_MAP = {
    "healthy_leaves": 0, "healthystem": 0, "cinnamon_stem": 0, "unlabeled": 0, 
    
    "leaf_spot_disease": -1,
    "leaf-spot-low-stage": 1,
    "lowstage_leafspot_cinnamon": 1,
    "leaf-spot-medium-stage": 2,
    "mediumstage_leafspot_cinnamon": 2,
    "leaf-spot-high-stage": 3,
    "highstage_leafspot_cinnamon": 3,
    
    "roughbark": 4,              
    
    "stripecanker": -1,
    "stripe-canker-low-stage": 5,
    "lowstage_stripecanker_cinnamon": 5,
    "stripe-canker-medium-stage": 6,
    "mediumstage_stripecanker_cinnamon": 6,
    "stripe-canker-high-stage": 7,
    "highstage_stripecanker_cinnamon": 7
}

OUTPUT_DIR = Path("Cinnamon_Master_Dataset")

def setup_directories():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR) 
    for split in ['train', 'test', 'val']:
        (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)

import pandas as pd
from pathlib import Path
import shutil

DATASET_PATHS = {
    "Roboflow_A": Path(r"D:\CINNAMON\Diseases\C1"),
    "Roboflow_B": Path(r"D:\CINNAMON\Diseases\C2"),
    "Roboflow_C": Path(r"D:\CINNAMON\Diseases\C3"),
    "Roboflow_D": Path(r"D:\CINNAMON\Diseases\C4"),
    "Roboflow_E": Path(r"D:\CINNAMON\Diseases\C5"),
    "Roboflow_F": Path(r"D:\CINNAMON\Diseases\C6"),
    "Kaggle_D": Path(r"D:\CINNAMON\Diseases\K")
}

def process_roboflow_set(dataset_name, path, master_map):
    all_data = []
    for split in ['train', 'valid', 'test']:
        target_split = 'val' if split == 'valid' else split
        csv_path = path / split / "_classes.csv"
        if not csv_path.exists(): continue
            
        df = pd.read_csv(csv_path)
        class_columns = [col for col in df.columns if col not in ['filename', 'width', 'height', 'class']]

        for _, row in df.iterrows():
            img_name = row['filename']
            active_classes = [col for col in class_columns if row[col] == 1]
            label = active_classes[0].strip().lower().replace(" ", "-") if active_classes else "unlabeled"
            master_id = master_map.get(label, -1)
            
            if master_id == -1: continue 
                
            src_img = path / split / img_name
            dest_img = OUTPUT_DIR / target_split / "images" / f"{dataset_name}_{img_name}"
            
            if src_img.exists():
                shutil.copy(src_img, dest_img)
                all_data.append({"filename": f"{dataset_name}_{img_name}", "class_id": master_id, "label": label, "split": target_split})
    return all_data

def process_kaggle_set(path, master_map):
    all_data = []
    for folder_name in ["RoughBark"]: 
        src_folder = path / folder_name
        label = folder_name.lower().replace(" ", "-")
        master_id = master_map.get(label, -1)
        if master_id == -1 or not src_folder.exists(): continue
            
        images = list(src_folder.glob("*.jpg")) + list(src_folder.glob("*.png"))
        for i, img_path in enumerate(images):
            split = 'train' if i < 0.8 * len(images) else ('val' if i < 0.9 * len(images) else 'test')
            dest_name = f"Kaggle_{img_path.name}"
            shutil.copy(img_path, OUTPUT_DIR / split / "images" / dest_name)
            all_data.append({"filename": dest_name, "class_id": master_id, "label": label, "split": split})
    return all_data

setup_directories()
master_records = []

print("Checking paths and aggregating...")
for name, path in DATASET_PATHS.items():
    if not path.exists():
        print(f"CRITICAL ERROR: Path for {name} does not exist at {path.absolute()}")
        continue

    if "Roboflow" in name:
        records = process_roboflow_set(name, path, MASTER_CLASS_MAP)
        print(f"Processed {name}: Found {len(records)} images.")
        master_records.extend(records)
    elif name == "Kaggle_D":
        records = process_kaggle_set(path, MASTER_CLASS_MAP)
        print(f"Processed Kaggle_D: Found {len(records)} images.")
        master_records.extend(records)

if not master_records:
    print("\nERROR: No data was found. Please check your folder paths and filenames.")
    master_df = pd.DataFrame(columns=["filename", "class_id", "label", "split"])
else:
    master_df = pd.DataFrame(master_records)

if not master_df.empty:
    for split in ['train', 'test', 'val']:
        if 'split' in master_df.columns:
            split_df = master_df[master_df['split'] == split]
            split_df.to_csv(OUTPUT_DIR / split / "metadata.csv", index=False)
            print(f"Saved {split} metadata to {OUTPUT_DIR / split}")
    
    print("\n--- VALIDATION REPORT ---")
    print(f"Total Images Aggregated: {len(master_df)}")
    print("\nTrue Class Distribution:")
    
    DISPLAY_MAP = {
        0: "Healthy", 
        1: "LS-Low", 
        2: "LS-Med", 
        3: "LS-High", 
        4: "RoughBark", 
        5: "SC-Low", 
        6: "SC-Med", 
        7: "SC-High"}
    
    master_df['Display_Name'] = master_df['class_id'].map(DISPLAY_MAP)
    
    print(master_df['Display_Name'].value_counts().sort_index())
else:
    print("Execution halted: master_df is empty.")

Checking paths and aggregating...
Processed Roboflow_A: Found 428 images.
Processed Roboflow_B: Found 631 images.
Processed Roboflow_C: Found 58 images.
Processed Roboflow_D: Found 1357 images.
Processed Roboflow_E: Found 227 images.
Processed Roboflow_F: Found 2951 images.
Processed Kaggle_D: Found 158 images.
Saved train metadata to Cinnamon_Master_Dataset\train
Saved test metadata to Cinnamon_Master_Dataset\test
Saved val metadata to Cinnamon_Master_Dataset\val

--- VALIDATION REPORT ---
Total Images Aggregated: 5810

True Class Distribution:
Display_Name
Healthy       997
LS-High       425
LS-Low        668
LS-Med        408
RoughBark     516
SC-High       485
SC-Low       1024
SC-Med       1287
Name: count, dtype: int64
